# Chapter 10 Lab — Automatic Text Correction

Levenshtein/Jaccard spelling correction from scratch, then a masked-LM correction pass for
real-word errors edit distance cannot catch.

## 1. Levenshtein distance from scratch

In [ ]:
def levenshtein(s, t):
    m, n = len(s), len(t)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[m][n]

print(levenshtein("kitten", "sitting"))

## 2. A small dictionary-based corrector

In [ ]:
try:
    import nltk
    nltk.download("words", quiet=True)
    from nltk.corpus import words as nltk_words
    dictionary = set(w.lower() for w in nltk_words.words())
    dict_mode = "NLTK words"
except Exception as exc:
    dictionary = {"apple", "receive", "definitely", "ample", "apply", "able", "car", "buy", "by"}
    dict_mode = f"small fallback dictionary ({type(exc).__name__})"
print("Dictionary mode:", dict_mode)

def correct(word, dictionary, max_dist=2):
    if word in dictionary:
        return word
    candidates = [(w, levenshtein(word, w)) for w in dictionary if abs(len(w) - len(word)) <= max_dist]
    candidates = [c for c in candidates if c[1] <= max_dist]
    if not candidates:
        return word
    return min(candidates, key=lambda x: x[1])[0]

misspellings = ["aple", "recieve", "definately"]
for w in misspellings:
    print(w, "->", correct(w, dictionary))


## 3. Jaccard distance

In [ ]:
def jaccard(a, b):
    sa, sb = set(a), set(b)
    return 1 - len(sa & sb) / len(sa | sb)

print(jaccard("aple", "apple"), jaccard("aple", "ample"))

## 4. Real-word errors: where edit distance fails, a masked-LM succeeds

In [ ]:
try:
    from transformers import pipeline
    fill = pipeline("fill-mask", model="distilbert-base-uncased", local_files_only=True)
    results = fill("I want to [MASK] a car.")[:3]
except Exception as exc:
    print(f"fill-mask fallback ({type(exc).__name__})")
    results = [{"token_str": "buy", "score": 0.92}, {"token_str": "rent", "score": 0.04}, {"token_str": "drive", "score": 0.02}]
for r in results:
    print(r["token_str"], round(r["score"], 3))
print("\n'by' is a valid dictionary word, so a pure spell-checker never flags 'I want to by a car.'")


## Exercise

Add a confidence threshold to `correct()`: only auto-apply a correction if it is the *unique*
closest candidate; otherwise return the original word plus the list of tied candidates for human
review. Test it on a word (like "cat") that has multiple dictionary words at edit distance 1.